# KoBART Summarizer 재학습 (Google Colab)

lotte-insight 프로젝트 — `training/train_summarizer.py` Colab 실행용 노트북

**학습 타깃:** `event_summary` 평문 한국어 문장 (JSON 아님)  
**평가 지표:** `eval_loss` (best model 선택 기준), `char_f1`, `exact_match`

In [ ]:
# 1. GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available — 런타임을 GPU로 변경하세요')
print('CUDA:', torch.version.cuda)

In [ ]:
# 2. 레포 클론 및 의존성 설치
# GitHub URL을 본인 레포로 교체하시오
GITHUB_REPO_URL = 'https://github.com/JoeYunHa/Lotte_Insight'

!git clone {GITHUB_REPO_URL} /content/lotte-insight
%cd /content/lotte-insight/training
!pip install -q -r requirements.txt

In [ ]:
# 3. Google Drive 마운트 및 학습 데이터 복사
# 사전 준비: 아래 두 파일을 Drive의 MyDrive/lotte-insight-data/ 에 업로드
#   - labeled_titles.csv
#   - labeled_players.csv
import os
import shutil
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/lotte-insight-data'
DATA_DIR = '/content/lotte-insight/training/data'
os.makedirs(DATA_DIR, exist_ok=True)

for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    src = f'{DRIVE_DATA_DIR}/{fname}'
    dst = f'{DATA_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'복사 완료: {fname}')
    else:
        print(f'[MISSING] Drive에 파일 없음: {src}')

print()
for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    lotte = df[df['is_lotte_related'].astype(str).str.lower() == 'true']
    with_summary = lotte['event_summary'].fillna('').astype(str).str.strip().ne('').sum()
    print(f'{fname}: 전체={len(df)}행  lotte={len(lotte)}행  event_summary 있음: {with_summary}행')

print('\n샘플 확인 (event_summary 평문):')
for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    samples = df[df['event_summary'].fillna('').astype(str).str.strip().ne('')]['event_summary'].head(2)
    for s in samples:
        print(f'  {repr(s[:80])}')

In [ ]:
# 4. 학습 실행
# 타깃: event_summary 평문 (JSON 아님)
# best model 기준: eval_loss 최소
!python train_summarizer.py \
    --epochs 5 \
    --batch 8 \
    --max-source-len 256 \
    --max-target-len 192 \
    --num-beams 4 \
    --early-stopping-patience 2

In [ ]:
# 5. 학습 결과 확인
import json

state_path = '/content/lotte-insight/training/models/summarizer_kobart/trainer_state.json'
with open(state_path) as f:
    state = json.load(f)

print(f'best_metric (eval_loss): {state["best_metric"]:.4f}')
print(f'best_step: {state["best_global_step"]}')
print(f'총 학습 step: {state["global_step"]}')

# 에폭별 eval 지표 요약
print('\n에폭별 평가 지표:')
for entry in state['log_history']:
    if 'eval_loss' in entry:
        char_f1 = entry.get('eval_char_f1', entry.get('char_f1', '-'))
        print(f"  epoch={entry['epoch']:.0f}  eval_loss={entry['eval_loss']:.4f}  char_f1={char_f1}")

In [ ]:
# 6. 빠른 추론 테스트 (학습 직후 품질 확인)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

MODEL_DIR = '/content/lotte-insight/training/models/summarizer_kobart'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
model.eval()

SAMPLES = [
    'title: 롯데 나균안, 시즌 5승 달성…선발 로테이션 안정화\ndescription: 나균안이 두산전 6이닝 2실점 호투로 시즌 5승을 따냈다.\ntopic_label: MATCH_RELATED',
    'title: 롯데 전준우 햄스트링 부상…2주 결장 예상\ndescription: 전준우가 경기 중 햄스트링 부상으로 1군 엔트리에서 말소됐다.\ntopic_label: INJURY_ROSTER',
]

for source in SAMPLES:
    inputs = tokenizer(source, max_length=256, truncation=True, return_tensors='pt')
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=192, num_beams=4, no_repeat_ngram_size=3)
    result = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    print(f'입력: {source.splitlines()[0]}')
    print(f'요약: {result}')
    print()

In [ ]:
# 7. 학습된 모델을 Google Drive에 저장
import shutil
from google.colab import drive
drive.mount('/content/drive')

LOCAL_MODEL_DIR = '/content/lotte-insight/training/models/summarizer_kobart'
DRIVE_MODEL_DIR = '/content/drive/MyDrive/lotte-insight-data/models/summarizer_kobart'

if os.path.exists(LOCAL_MODEL_DIR):
    # 이전 저장본 제거 후 복사
    if os.path.exists(DRIVE_MODEL_DIR):
        shutil.rmtree(DRIVE_MODEL_DIR)
    shutil.copytree(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR)
    files = os.listdir(DRIVE_MODEL_DIR)
    print(f'저장 완료: {DRIVE_MODEL_DIR}')
    print('파일 목록:', [f for f in files if not f.startswith('checkpoint')])
else:
    print('[ERROR] 모델 디렉토리 없음 — 학습 실패 여부 확인 필요')

In [ ]:
# 8. (선택) 평가만 실행할 경우
# !python train_summarizer.py --eval-only